# 评估指标：如何衡量模型好坏
> 难度标签：中级 | 预计时长：10-20分钟 | 前置知识：无学习经验


> 所属模块：12_模型评估与选择 | 源文件：评估指标.py | 核心功能：准确率、精确率、召回率、F1、R^2、MSE

## 概述

不同任务需要不同的评估指标。分类：准确率、精确率、召回率、F1。回归：MSE、RMSE、MAE、R^2。

**导入必要的库**

In [ ]:
# 确保中文输出正常（Windows 环境兼容）
import sys
try:
    sys.stdout.reconfigure(encoding='utf-8')
except AttributeError:
    pass

import numpy as np
from sklearn.datasets import make_classification, make_regression
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.ensemble import RandomForestClassifier
# --- 导入库 ---
from sklearn.preprocessing import label_binarize

## 数学原理

### 1. 分类指标

**准确率（Accuracy）**：

$$\text{Acc} = \frac{1}{n}\sum_{i=1}^{n}\mathbb{I}[\hat{y}_i = y_i] = \frac{TP + TN}{TP + TN + FP + FN}$$

**精确率（Precision）**：预测为正的样本中真正为正的比例

$$P = \frac{TP}{TP + FP}$$

**召回率（Recall / Sensitivity）**：真实正样本中被正确预测的比例

$$R = \frac{TP}{TP + FN}$$

**F1 值**：精确率和召回率的调和平均

$$F1 = \frac{2PR}{P + R} = \frac{2TP}{2TP + FP + FN}$$

**宏平均（Macro）**：各类别指标的简单平均

$$\text{Macro-F1} = \frac{1}{C}\sum_{c=1}^{C} F1_c$$

**加权平均（Weighted）**：按类别样本数加权

$$\text{Weighted-F1} = \sum_{c=1}^{C}\frac{n_c}{n} F1_c$$

### 2. 回归指标

**MSE（均方误差）**：$\text{MSE} = \frac{1}{n}\sum_{i=1}^{n}(y_i - \hat{y}_i)^2$

**RMSE（均方根误差）**：$\text{RMSE} = \sqrt{\text{MSE}}$

**MAE（平均绝对误差）**：$\text{MAE} = \frac{1}{n}\sum_{i=1}^{n}|y_i - \hat{y}_i|$

**$R^2$（决定系数）**：

$$R^2 = 1 - \frac{\sum(y_i - \hat{y}_i)^2}{\sum(y_i - \bar{y})^2} = 1 - \frac{\text{MSE}}{\text{Var}(y)}$$

$R^2 = 1$ 表示完美拟合，$R^2 = 0$ 表示等价于预测均值。

### 3. 代码与数学的对应

| 代码 | 数学含义 |
|------|----------|
| `accuracy_score(y_true, y_pred)` | $\text{Acc} = \frac{TP+TN}{n}$ |
| `precision_score(y_true, y_pred, average="macro")` | $\frac{1}{C}\sum_c P_c$ |
| `recall_score(y_true, y_pred, average="weighted")` | $\sum_c \frac{n_c}{n} R_c$ |
| `f1_score(y_true, y_pred)` | $F1 = \frac{2PR}{P+R}$ |
| `mean_squared_error(y_true, y_pred)` | $\frac{1}{n}\sum(y-\hat{y})^2$ |
| `r2_score(y_true, y_pred)` | $R^2 = 1 - \frac{\text{MSE}}{\text{Var}(y)}$ |
| `classification_report()` | 输出 P, R, F1 的完整表格 |

### 分类评估指标

在分类任务上训练模型并评估性能。

In [ ]:
print("=" * 60)
print("分类评估指标")
print("=" * 60)

X_cls, y_cls = make_classification(
    n_samples=500, n_features=10, n_informative=5,
    n_classes=3, n_clusters_per_class=1, random_state=42
)

X_train, X_test, y_train, y_test = train_test_split(
    X_cls, y_cls, test_size=0.3, random_state=42, stratify=y_cls
)

model_cls = RandomForestClassifier(n_estimators=100, random_state=42)
model_cls.fit(X_train, y_train)
y_pred = model_cls.predict(X_test)
y_proba = model_cls.predict_proba(X_test)

# 手动计算混淆矩阵
n_classes = 3
cm = np.zeros((n_classes, n_classes), dtype=int)
for true, pred in zip(y_test, y_pred):
    cm[true, pred] += 1

print("混淆矩阵 (手动计算):")
print(f"{'预测->':>10}", end="")
for j in range(n_classes):
    print(f"  类{j}  ", end="")
print()
# --- 循环处理 ---
for i in range(n_classes):
    print(f"  真实类{i} ", end="")
    for j in range(n_classes):
        print(f"  {cm[i, j]:>3}  ", end="")
    print()

# 1. 准确率 (Accuracy)
from sklearn.metrics import accuracy_score
accuracy = accuracy_score(y_test, y_pred)
manual_accuracy = np.trace(cm) / cm.sum()
print(f"\n1. 准确率 (Accuracy)")
print(f"   sklearn: {accuracy:.4f}")
# --- 输出结果 ---
print(f"   手动计算: {manual_accuracy:.4f}")
print(f"   含义: 正确预测的样本数 / 总样本数")

# 2. 精确率、召回率、F1 (逐类别)
from sklearn.metrics import precision_score, recall_score, f1_score

print(f"\n2. 精确率 (Precision) / 召回率 (Recall) / F1")
for avg in ['macro', 'micro', 'weighted']:
    p = precision_score(y_test, y_pred, average=avg)
    r = recall_score(y_test, y_pred, average=avg)
    f1 = f1_score(y_test, y_pred, average=avg)
# --- 输出结果 ---
    print(f"   {avg:>8}: 精确率={p:.4f}, 召回率={r:.4f}, F1={f1:.4f}")

# 逐类别
print(f"\n   逐类别明细:")
print(f"   {'类别':>6} {'精确率':>8} {'召回率':>8} {'F1':>8} {'支持数':>8}")
for i in range(n_classes):
    p_i = precision_score(y_test, y_pred, average=None)[i]
    r_i = recall_score(y_test, y_pred, average=None)[i]
# --- 继续 ---
    f1_i = f1_score(y_test, y_pred, average=None)[i]
    support = np.sum(y_test == i)
    print(f"   类{i:>4} {p_i:>8.4f} {r_i:>8.4f} {f1_i:>8.4f} {support:>8}")

# 3. AUC (二分类和多分类)
from sklearn.metrics import roc_auc_score

print(f"\n3. AUC (Area Under ROC Curve)")

# 二分类简化示例
y_binary = (y_test == 0).astype(int)
y_proba_binary = y_proba[:, 0]
auc_binary = roc_auc_score(y_binary, y_proba_binary)
print(f"   二分类AUC (类0 vs 其他): {auc_binary:.4f}")

# 多分类 AUC (one-vs-rest)
auc_ovr = roc_auc_score(y_test, y_proba, multi_class='ovr', average='macro')
auc_ovo = roc_auc_score(y_test, y_proba, multi_class='ovo', average='macro')
print(f"   多分类AUC (One-vs-Rest, macro): {auc_ovr:.4f}")
print(f"   多分类AUC (One-vs-One, macro):  {auc_ovo:.4f}")

### 回归评估指标

在回归任务上拟合模型，观察预测值与真实值的偏差。

In [ ]:
print("\n" + "=" * 60)
print("回归评估指标")
print("=" * 60)

X_reg, y_reg = make_regression(
    n_samples=300, n_features=5, n_informative=3,
    noise=20, random_state=42
)

X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(
    X_reg, y_reg, test_size=0.3, random_state=42
)

model_reg = LinearRegression()
model_reg.fit(X_train_r, y_train_r)
y_pred_r = model_reg.predict(X_test_r)
residuals = y_test_r - y_pred_r

# 1. 均方误差 (MSE)
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

mse = mean_squared_error(y_test_r, y_pred_r)
manual_mse = np.mean(residuals ** 2)
print(f"1. 均方误差 (MSE)")
print(f"   sklearn: {mse:.4f}")
print(f"   手动计算: {manual_mse:.4f}")

# 2. 均方根误差 (RMSE)
rmse = np.sqrt(mse)
print(f"\n2. 均方根误差 (RMSE)")
print(f"   RMSE = {rmse:.4f}")
print(f"   与目标变量同单位，更直观")

# 3. 平均绝对误差 (MAE)
mae = mean_absolute_error(y_test_r, y_pred_r)
manual_mae = np.mean(np.abs(residuals))
print(f"\n3. 平均绝对误差 (MAE)")
print(f"   sklearn: {mae:.4f}")
print(f"   手动计算: {manual_mae:.4f}")

# 4. R-squared (决定系数)
r2 = r2_score(y_test_r, y_pred_r)
manual_r2 = 1 - np.sum(residuals ** 2) / np.sum((y_test_r - y_test_r.mean()) ** 2)
print(f"\n4. R-squared (决定系数)")
print(f"   sklearn: {r2:.4f}")
print(f"   手动计算: {manual_r2:.4f}")
# --- 输出结果 ---
print(f"   含义: 模型解释了目标变量{r2*100:.1f}%的方差")

# 5. 残差分析
print(f"\n5. 残差分析")
print(f"   残差均值: {residuals.mean():.4f} (应接近0)")
print(f"   残差标准差: {residuals.std():.4f}")
print(f"   残差最小值: {residuals.min():.4f}")
print(f"   残差最大值: {residuals.max():.4f}")

### 综合对比

对比不同模型或配置的性能差异。

In [ ]:
print("\n" + "=" * 60)
print("指标选择指南")
print("=" * 60)
print("""
分类任务:
# --- 继续 ---
  - 准确率: 类别平衡时使用
  - 精确率: 关注误报（FP）时使用（如垃圾邮件检测）
  - 召回率: 关注漏报（FN）时使用（如疾病诊断）
  - F1: 精确率和召回率的调和平均，类别不平衡时使用
  - AUC: 不依赖阈值，评估整体排序能力

回归任务:
  - MSE/RMSE: 对大误差敏感，适合需要惩罚大偏差的场景
  - MAE: 对异常值更鲁棒
  - R-squared: 无量纲，便于跨数据集比较
""")

## 关键代码解释

```python
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.metrics import mean_squared_error, r2_score
```

## 注意事项

1. 类别不平衡时准确率不可靠
2. 选择与业务目标一致的指标
3. 单一指标不够，通常需要看多个

## 总结与延伸

以上代码展示了 **评估指标** 的完整流程。

**解读要点**：
- 关注 **ROC 曲线** 下的面积（AUC）
- 观察 **交叉验证** 各折的方差
- 对比不同评估指标的适用场景

**进一步思考**：尝试修改关键参数（如正则化强度、学习率、树深度等），观察结果如何变化。

### 延伸阅读

- **AUC-ROC**：不受阈值影响的分类指标
- **PR 曲线**：严重不平衡时比 ROC 更有信息量
- **自定义指标**：`make_scorer`
